# 00 Environment and data audit

**Question.** Is this machine ready for study v2 (environment, data, weights, split, provenance)?

**Type.** Inspection only. Nothing is trained and no model is run. **Prerequisites.** `uv sync --extra dev`.
Outputs are not stored in this notebook; run it to see them.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
from ct_iqa.experiment.metadata import environment_info, git_info
from ct_iqa.data.splits import load_manifest
from ct_iqa.models.weights import sha256_file
print(environment_info("cpu")); print(git_info(REPO))

## Labels and frozen split
Checks label counts, score range and that the manifest partitions are disjoint and cover the labeled images.

In [ ]:
labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
test_labels = json.loads(Path("data/labels/ldct_iqac/test.json").read_text())
print(len(labels), len(test_labels), min(labels.values()), max(labels.values()))
manifest = load_manifest("experiments/study_v2/protocol/split_manifest.json", labels)
print({k: len(manifest[k]) for k in ("train", "validation", "calibration")}, manifest["manifest_sha256"])

## Weights and historical checkpoints
Verifies the converted RadImageNet weights against the recorded SHA-256 and reports which historical checkpoints exist.

In [ ]:
expected = dict(l.split()[::-1] for l in Path("weights/pretrained/radimagenet/resnet50/CHECKSUMS.sha256").read_text().splitlines())
for name, digest in expected.items():
    p = Path("weights/pretrained/radimagenet/resnet50") / name.lstrip("*")
    print(name, "present" if p.is_file() else "MISSING", "hash OK" if p.is_file() and sha256_file(p) == digest else "hash not verified")
for p in sorted(Path("experiments").glob("00*/**/checkpoint/*.pt")):
    print("checkpoint:", p)
print("Structural compatibility: docs/audit/checkpoint-compatibility.md")

## Raw image availability
Counts files only; images are not read.

In [ ]:
for d in ("data/raw/ldct_iqac/train/image", "data/raw/ldct_iqac/test/images"):
    print(d, len(list(Path(d).glob("*.tif*"))) if Path(d).is_dir() else "MISSING")